# LanceDB vector database

In [ ]:
import lancedb

# in python script: Path(__file__).parent / "vector_database"
db = lancedb.connect(uri="vector_database")
db

In [ ]:
db.uri

## Read in data

In [ ]:
import json 
with open("data/animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())

data

## Create table

In [ ]:
db.create_table("animals", exist_ok=True, data=data)


In [ ]:
db.list_tables()

In [ ]:
db["animals"]

In [ ]:
db["animals"].head()

convert to a classic dataframe

In [ ]:
df_animals = db["animals"].to_pandas()
df_animals

In [ ]:
df_animals.iloc[2]["text"], df_animals.iloc[2]["vector"]

to add more data

In [ ]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]},
]

db["animals"].add(more_data)

In [ ]:
db["animals"].to_pandas()

## Create empty table
- create an empty table first and then place in data 
- need to provide a schema

In [ ]:
from lancedb.pydantic import LanceModel

class EmployeeSchema(LanceModel):
    first_name: str 
    last_name: str 
    salary: int

db.create_table(name = "employees", schema=EmployeeSchema, exist_ok=True)

In [ ]:
data = [{"first_name": "Bibbi", "last_name": "Babblarna", "salary": 1000}]
db["employees"].add(data)

In [ ]:
db["employees"].to_pandas()

In [ ]:
db.list_tables()

In [ ]:
db.drop_table("employees")

In [ ]:
db.list_tables()

## Vector search

ANN - approximate nearest neighbour for vector search

1. send in a query vector directly and search
    - this requires that we embed our query first using same embedding as what was used in the knowledge base
2. send in a text and let lancedb automatically embed it and search


In [ ]:
db["animals"].to_pandas()

In [ ]:
# assume that we embed our question using same embedding model as the one for animals
# question about elephant
query_vector = [0.9, 0.2, 0.5]

db["animals"].search(query_vector).limit(4).to_pandas()

## Embeddings API

- let lancedb embed our documents automatically
- let lancedb embed our query automatically and search using natural language



In [19]:

from dotenv import load_dotenv

load_dotenv()

True

In [20]:
from lancedb.pydantic import Vector
from lancedb.embeddings import get_registry

model = get_registry().get("gemini-text").create(name="gemini-embedding-001")

model


GeminiText(max_retries=7, name='gemini-embedding-001', query_task_type='retrieval_query', source_task_type='retrieval_document')

In [21]:
embeddings = model.generate_embeddings("Why are SQL good at relationships? Because they are relational")

In [22]:
import numpy as np 
np.array(embeddings).shape

(62, 3072)

In [ ]:
class JokeModel(LanceModel):
    joke: str = model.SourceField() # input to embedding function
    embedding: Vector(3072) = model.VectorField() # computed embedding in this column

db.create_table("jokes", schema=JokeModel, exist_ok=True) 

In [ ]:
import pandas as pd 
with open("data/jokes.json", "r") as file:
    jokes_data = json.loads(file.read())

df_jokes = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)
df_jokes.head()

In [ ]:
db["jokes"].add(df_jokes)

In [ ]:
db["jokes"].to_pandas().head()

In [ ]:
db["jokes"].to_pandas().iloc[2]["embedding"].shape

In [ ]:
db["jokes"].search("snakey joke").limit(3).to_pandas()